# Búsqueda y selección de modelos

Este notebook desarrolla el objetivo específico 2 del proyecto: desarrollar y comparar modelos de estimación ordinal del SII basados en indicadores físicos y demográficos, frente a un modelo de referencia, mediante validación cruzada estratificada y el QWK. Se comparan cuatro algoritmos —un modelo lineal regularizado, una SVM con kernel radial, un bosque aleatorio y gradient boosting— en tres formulaciones del SII —clasificación multiclase, descomposición ordinal de Frank y Hall y regresión con umbrales—, con la estrategia de tratamiento de los faltantes y la ponderación de los niveles como hiperparámetros de la validación interna.

El análisis exploratorio (`01_eda.ipynb`) definió el conjunto analítico: los participantes con SII registrado, las reglas de limpieza de valores inválidos, las variables excluidas por cobertura o redundancia, el resumen de la actigrafía y cuatro conjuntos anidados de predictores (referencia, principal sin actigrafía, principal y complementario). Los experimentos iniciales (`02_experimentos_iniciales.ipynb`) evaluaron dos configuraciones fijas en la formulación multiclase: no mostraron evidencia de mejora del conjunto principal frente a la referencia (H2) ni de la actigrafía (H3), y las dos configuraciones trataron el nivel severo de forma opuesta. Por eso la búsqueda completa amplía los algoritmos, las formulaciones y la ponderación, y vuelve a evaluar H2 y H3 con la configuración que resulte seleccionada.

El notebook se organiza así:

1. Datos y conjuntos de variables.
2. Diseño de la búsqueda.
3. Estimación del costo de cómputo.
4. Búsqueda completa.
5. Hiperparámetros elegidos y revisión de los bordes de la grilla.
6. Selección de la configuración.
7. Desempeño por nivel de la configuración seleccionada.
8. H2 y H3 con la configuración seleccionada.
9. Comparación en la subpoblación con actigrafía.
10. Configuración guardada y conclusiones.

In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from piu_severity.config import FIGURES_DIR, INTERIM_DATA_DIR, RANDOM_SEED
from piu_severity.data.clean import clean_values
from piu_severity.data.load import list_actigraphy_ids, load_train
from piu_severity.evaluation.metrics import SII_LEVELS, as_levels
from piu_severity.evaluation.persistence import (
    CACHE_VERSION,
    directory_size,
    evaluation_fields,
    run_nested_cv_cached,
)
from piu_severity.evaluation.selection import (
    EDGE_RULE_MIN_FOLDS,
    HOLM_FAMILIES,
    MINIMUM_RELEVANT_QWK_DIFFERENCE,
    SELECTION_ALPHA,
    UNDERESTIMATION_TIE_TOLERANCE,
)
from piu_severity.evaluation.validation import (
    N_INNER_SPLITS,
    N_OUTER_REPEATS,
    N_OUTER_SPLITS,
    outer_splits,
    split_level_counts,
)
from piu_severity.features.actigraphy import (
    load_actigraphy_summary,
    usable_actigraphy_mask,
)
from piu_severity.features.build import build_analytic_dataset
from piu_severity.features.groups import (
    TARGET_COLUMN,
    experiment_feature_sets,
    is_excluded,
    is_instrument_season,
)
from piu_severity.models.pipelines import (
    ALGORITHMS,
    BOOSTING_MAX_ITER,
    BOOSTING_N_ITER_NO_CHANGE,
    BOOSTING_VALIDATION_FRACTION,
    CONVERGENCE_WARNING_MAX_SHARE,
    CUT_BOOSTING_RANDOM_CANDIDATES,
    CUT_FOREST_N_ESTIMATORS,
    FOREST_CORES_PER_WORKER,
    FOREST_N_ESTIMATORS,
    FORMULATIONS,
    LINEAR_MAX_ITER,
    MAX_CUT_LEVEL,
    MAX_ITER_INCREASE,
    MEMORY_BUDGET_FRACTION,
    MEMORY_SAFETY_FACTOR,
    RESERVED_CORES,
    SEARCH_BUDGET_HOURS,
    available_workers,
    choose_cut_level,
    convergence_warning_share,
    make_search,
    max_iter_factor_for,
    memory_budget_gb,
    n_candidates,
    n_forest_trees,
    project_search_hours,
    search_grid,
    total_memory_gb,
    workers_within_memory,
)
from piu_severity.visualization.figures import save_figure

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
sns.set_theme(style="whitegrid")

# Figuras del notebook y carpetas de los dos cachés. Ambas carpetas están bajo
# data/interim, que git ignora, porque guardan datos transformados y predicciones
# por participante.
FIGURAS = FIGURES_DIR / "03_busqueda_modelos"
CARPETA_NOTEBOOK = INTERIM_DATA_DIR / "03_busqueda_modelos"
# Caché de resultados: un archivo por configuración, conjunto, repetición y partición.
CARPETA_CACHE = CARPETA_NOTEBOOK / "cache"
# Caché de Pipeline(memory=...): evita reajustar el preprocesamiento entre candidatos.
# Incluye la versión del caché porque joblib no detecta cambios en el código propio.
CARPETA_MEMORIA = CARPETA_NOTEBOOK / "pipeline_memory" / f"v{CACHE_VERSION}"

# Con True se recalculan todas las evaluaciones aunque exista un resultado válido.
RECALCULAR = False

# Recursos del equipo en el que se ejecuta el notebook: los procesos en paralelo
# y el presupuesto de memoria se calculan a partir de ellos.
N_NUCLEOS = os.cpu_count()
MEMORIA_TOTAL_GB = total_memory_gb()

NIVELES_SII = {0: "ninguno", 1: "leve", 2: "moderado", 3: "severo"}
NOMBRES_CONJUNTOS = {
    "reference": "referencia",
    "reference_with_actigraphy": "referencia con actigrafía",
    "main_without_actigraphy": "principal sin actigrafía",
    "main": "principal",
    "complementary": "complementario",
}
NOMBRES_ALGORITMOS = {
    "linear": "lineal regularizado",
    "svm": "SVM con kernel radial",
    "random_forest": "bosque aleatorio",
    "gradient_boosting": "gradient boosting",
}
NOMBRES_FORMULACIONES = {
    "threshold_regression": "regresión con umbrales",
    "multiclass": "multiclase",
    "frank_hall": "Frank y Hall",
}
NOMBRES_ESTRATEGIAS = {
    "native": "nativa",
    "simple": "simple con indicadores",
    "iterative": "iterativa",
}
# Las doce configuraciones de la búsqueda, en el orden de complejidad del paquete.
CONFIGURACIONES = [(a, f) for a in ALGORITHMS for f in FORMULATIONS]

## 1. Datos y conjuntos de variables

La búsqueda usa el mismo conjunto analítico de los notebooks anteriores, construido con las funciones del paquete: los participantes con SII registrado, con las reglas de limpieza del análisis exploratorio y el resumen de las series de actigrafía. Antes de modelar, esta sección verifica que el conjunto reproduce las cifras de la Entrega 2 (número de participantes, de casos severos, de predictores por conjunto y de participantes con una serie de actigrafía utilizable), que ningún conjunto incluye las columnas del PCIAT, el identificador o las temporadas de cada instrumento, y construye las particiones externas que comparten todas las configuraciones.

In [3]:
# Participantes con SII y conjunto analítico, con las mismas funciones del paquete
# que usan los notebooks 01 y 02.
train = load_train()
etiquetados = train[train[TARGET_COLUMN].notna()].reset_index(drop=True)
analitico = build_analytic_dataset(
    clean_values(etiquetados), load_actigraphy_summary(list_actigraphy_ids())
)
y = as_levels(analitico[TARGET_COLUMN])
conjuntos = experiment_feature_sets(analitico.columns)

# Cifras de la Entrega 2 que este conjunto debe reproducir.
N_ETIQUETADOS_ENTREGA_2 = 2_736
N_SEVEROS_ENTREGA_2 = 34
N_PREDICTORES_ENTREGA_2 = {
    "reference": 3,
    "main": 36,
    "main_without_actigraphy": 33,
    "complementary": 39,
}
assert len(analitico) == N_ETIQUETADOS_ENTREGA_2
assert analitico["id"].is_unique
assert int((y == 3).sum()) == N_SEVEROS_ENTREGA_2
assert {nombre: len(c) for nombre, c in conjuntos.items()} == N_PREDICTORES_ENTREGA_2

# Ningún conjunto incluye el PCIAT, el identificador ni las temporadas por instrumento,
# y los conjuntos están anidados.
assert not any(
    is_excluded(c) or is_instrument_season(c)
    for columnas in conjuntos.values()
    for c in columnas
)
assert (
    set(conjuntos["reference"])
    <= set(conjuntos["main_without_actigraphy"])
    <= set(conjuntos["main"])
    <= set(conjuntos["complementary"])
)

distribucion_sii = (
    pd.Series(y)
    .map(NIVELES_SII)
    .value_counts()
    .reindex(NIVELES_SII.values())
    .rename_axis("SII")
    .to_frame("participantes")
)
distribucion_sii["%"] = (100 * distribucion_sii["participantes"] / len(y)).round(1)
distribucion_sii

,participantes,%
SII,,
ninguno,1594,58.3
leve,730,26.7
moderado,378,13.8
severo,34,1.2


In [4]:
# Subpoblación con una serie de actigrafía utilizable: la que se analiza aparte en la
# sección 9. Las cifras de la Entrega 2 se verifican aquí porque la búsqueda usa
# a todos los participantes y la ausencia de la serie se trata como un faltante más.
N_ACTIGRAFIA_ENTREGA_2 = 823
N_SEVEROS_ACTIGRAFIA_ENTREGA_2 = 6

con_actigrafia = usable_actigraphy_mask(analitico).to_numpy()
assert int(con_actigrafia.sum()) == N_ACTIGRAFIA_ENTREGA_2
assert int((y[con_actigrafia] == 3).sum()) == N_SEVEROS_ACTIGRAFIA_ENTREGA_2

composicion_conjuntos = pd.DataFrame(
    {
        "predictores": {
            NOMBRES_CONJUNTOS[nombre]: len(columnas)
            for nombre, columnas in conjuntos.items()
        },
        "incluye actigrafía": {
            NOMBRES_CONJUNTOS[nombre]: "sí" if any(
                c.startswith("Actigraphy-") for c in columnas
            ) else "no"
            for nombre, columnas in conjuntos.items()
        },
    }
).rename_axis("conjunto")

print(f"Participantes con SII: {len(analitico)}")
print(
    f"Participantes con serie de actigrafía utilizable: {int(con_actigrafia.sum())} "
    f"({100 * con_actigrafia.mean():.1f} %), de ellos "
    f"{int((y[con_actigrafia] == 3).sum())} severos"
)
composicion_conjuntos

Participantes con SII: 2736
Participantes con serie de actigrafía utilizable: 823 (30.1 %), de ellos 6 severos


,predictores,incluye actigrafía
conjunto,,
referencia,3,no
principal,36,sí
principal sin actigrafía,33,no
complementario,39,sí


In [5]:
# Particiones externas: validación cruzada estratificada por SII con cinco particiones,
# repetida cinco veces con la semilla 42 + r en la repetición r. Todas las
# configuraciones y conjuntos se evalúan sobre estas mismas particiones.
particiones = outer_splits(y)
assert len(particiones) == N_OUTER_SPLITS * N_OUTER_REPEATS

casos_por_particion = split_level_counts(particiones, y).rename(columns=NIVELES_SII)
print(
    f"{N_OUTER_REPEATS} repeticiones × {N_OUTER_SPLITS} particiones externas; "
    f"{N_INNER_SPLITS} particiones internas en cada búsqueda"
)
# Rango de casos de cada nivel en las 25 particiones de evaluación.
casos_por_particion.agg(["min", "max"]).rename_axis("casos por partición")

5 repeticiones × 5 particiones externas; 3 particiones internas en cada búsqueda


,ninguno,leve,moderado,severo
casos por partición,,,,
min,318,146,75,6
max,319,146,76,7


### Conclusión

El conjunto analítico reproduce las cifras de la Entrega 2:

- **Participantes y niveles.** Hay 2.736 participantes con SII: 58,3 % en el nivel ninguno, 26,7 % en leve, 13,8 % en moderado y 1,2 % en severo, que son 34 casos.
- **Predictores por conjunto.** La referencia tiene 3, el principal sin actigrafía 33, el principal 36 y el complementario 39.
- **Actigrafía.** 823 participantes (30,1 %) tienen una serie utilizable, y solo 6 de ellos son severos. Por eso la ausencia de la serie se trata en la búsqueda como un faltante más, y la comparación restringida a esa subpoblación se hace aparte, en la sección 9.
- **Exclusiones.** Ningún conjunto incluye columnas del PCIAT, el identificador ni las temporadas por instrumento, y los conjuntos están anidados.

Las 25 particiones externas conservan la proporción de cada nivel: cada partición de evaluación tiene entre 318 y 319 participantes del nivel ninguno, 146 del leve, entre 75 y 76 del moderado y entre 6 y 7 severos. Con tan pocos casos severos, un solo acierto cambia el recall severo de una partición en unos 15 puntos porcentuales. Por eso ese recall se calcula sobre las predicciones acumuladas de cada repetición, que cubren los 34 casos. Todas las configuraciones y conjuntos se evalúan sobre estas mismas particiones, lo que permite compararlos de forma pareada.

## 2. Diseño de la búsqueda

La búsqueda compara doce configuraciones: cuatro algoritmos por tres formulaciones del SII. Cada configuración es una búsqueda interna sobre una grilla que incluye la estrategia de faltantes, la intensidad de la ponderación de los niveles y los hiperparámetros del modelo. Todas las decisiones de esta sección —grillas, recortes por presupuesto, reglas de convergencia y de bordes, diferencia mínima relevante, tolerancia de empate, familias de pruebas y orden de complejidad— quedaron fijadas en el paquete antes de ejecutar la búsqueda, de modo que ninguna depende de sus resultados. Esta sección las reúne a partir del propio código.

In [6]:
def describir_valores(valores: list) -> str:
    """Resume los valores de un hiperparámetro: rango si son muchos, lista si son pocos."""
    numericos = all(isinstance(v, (int, float)) for v in valores)
    if numericos and len(valores) > 4:
        return f"{min(valores):g} a {max(valores):g} ({len(valores)} valores)"
    return ", ".join(f"{v:g}" if isinstance(v, (int, float)) else str(v) for v in valores)


# Parámetros de la grilla que no son del modelo: estrategia de faltantes y exponente
# de los pesos por nivel.
PARAMETROS_DE_TRATAMIENTO = ("preprocess__strategy", "model__weight_power")

filas = []
for algoritmo, formulacion in CONFIGURACIONES:
    grilla = search_grid(algoritmo, formulacion)
    filas.append(
        {
            "algoritmo": NOMBRES_ALGORITMOS[algoritmo],
            "formulación": NOMBRES_FORMULACIONES[formulacion],
            "estrategias de faltantes": ", ".join(
                NOMBRES_ESTRATEGIAS[e] for e in grilla["preprocess__strategy"]
            ),
            "α": ", ".join(f"{a:g}" for a in grilla["model__weight_power"]),
            # El nombre del hiperparámetro es la última parte de la ruta del pipeline.
            "hiperparámetros del modelo": "; ".join(
                f"{nombre.split('__')[-1]}: {describir_valores(valores)}"
                for nombre, valores in grilla.items()
                if nombre not in PARAMETROS_DE_TRATAMIENTO
            ),
            "candidatos": n_candidates(algoritmo, formulacion),
        }
    )

diseno_busqueda = pd.DataFrame(filas).set_index(["algoritmo", "formulación"])
diseno_busqueda

estrategias de faltantes  \
algoritmo             formulación                                                         
lineal regularizado   regresión con umbrales          simple con indicadores, iterativa   
                      multiclase                      simple con indicadores, iterativa   
                      Frank y Hall                    simple con indicadores, iterativa   
SVM con kernel radial regresión con umbrales          simple con indicadores, iterativa   
                      multiclase                      simple con indicadores, iterativa   
                      Frank y Hall                    simple con indicadores, iterativa   
bosque aleatorio      regresión con umbrales          simple con indicadores, iterativa   
                      multiclase                      simple con indicadores, iterativa   
                      Frank y Hall                    simple con indicadores, iterativa   
gradient boosting     regresión con umbrales  nativa, simple con indicadores, iterativa   
                      multiclase              nativa, simple con indicadores, iterativa   
                      Frank y Hall            nativa, simple con indicadores, iterativa   

                                                   α  \
algoritmo             formulación                      
lineal regularizado   regresión con umbrales  0.5, 1   
                      multiclase              0.5, 1   
                      Frank y Hall            0.5, 1   
SVM con kernel radial regresión con umbrales  0.5, 1   
                      multiclase              0.5, 1   
                      Frank y Hall            0.5, 1   
bosque aleatorio      regresión con umbrales  0.5, 1   
                      multiclase              0.5, 1   
                      Frank y Hall            0.5, 1   
gradient boosting     regresión con umbrales  0.5, 1   
                      multiclase              0.5, 1   
                      Frank y Hall            0.5, 1   

                                                                                                                                  hiperparámetros del modelo  \
algoritmo             formulación                                                                                                                              
lineal regularizado   regresión con umbrales                                                             alpha: 0.0001 a 10 (6 valores); l1_ratio: 0, 0.5, 1   
                      multiclase                                                                            C: 0.0001 a 100 (7 valores); l1_ratio: 0, 0.5, 1   
                      Frank y Hall                                                                          C: 0.0001 a 100 (7 valores); l1_ratio: 0, 0.5, 1   
SVM con kernel radial regresión con umbrales                                    C: 0.01 a 1000 (6 valores); gamma: 0.0001 a 1 (5 valores); epsilon: 0.1, 0.5   
                      multiclase                                                                   C: 0.01 a 1000 (6 valores); gamma: 0.0001 a 1 (5 valores)   
                      Frank y Hall                                                                 C: 0.01 a 1000 (6 valores); gamma: 0.0001 a 1 (5 valores)   
bosque aleatorio      regresión con umbrales                                               max_features: sqrt, 0.33, 0.66, 1; min_samples_leaf: 1, 5, 20, 50   
                      multiclase                                                           max_features: sqrt, 0.33, 0.66, 1; min_samples_leaf: 1, 5, 20, 50   
                      Frank y Hall                                                         max_features: sqrt, 0.33, 0.66, 1; min_samples_leaf: 1, 5, 20, 50   
gradient boosting     regresión con umbrales  learning_rate: 0.03, 0.1; max_leaf_nodes: 7, 15, 31; min_samples_leaf: 10, 30, 80; l2_regularization: 0, 1, 10   
                      multiclase              learning_rate: 0.03, 0.1; max_leaf_no

### Decisiones del diseño

**Formulaciones del SII.**
- *Multiclase nominal:* ignora el orden de los niveles y sirve como formulación base.
- *Frank y Hall:* tres clasificadores binarios estiman $P(\text{SII} > k)$ para $k = 0, 1, 2$. Como se entrenan por separado, se aplica el mínimo acumulado sobre $k$ antes de obtener por diferencias la probabilidad de cada nivel, y se asigna el nivel de mayor probabilidad. La SVM no produce probabilidades por sí misma, así que en esta formulación se calibra con una sigmoide ajustada en tres particiones internas estratificadas.
- *Regresión con umbrales:* el SII se estima como variable continua y se discretiza con tres umbrales ordenados que maximizan el QWK. Los umbrales se ajustan sobre predicciones fuera de pliegue obtenidas con tres particiones estratificadas por nivel dentro de los datos de entrenamiento de cada ajuste, de modo que la partición de evaluación no interviene. Como el QWK cambia de forma escalonada con los umbrales, se buscan por coordenadas sobre los cuantiles de las predicciones, desde dos puntos de partida.

**Tratamiento de los faltantes.** La estrategia es un hiperparámetro de la validación interna:
- *nativa:* los faltantes llegan al modelo; solo en gradient boosting, como establece la formulación;
- *simple:* mediana o moda, con un indicador de ausencia por instrumento;
- *iterativa:* imputación multivariada, sin indicadores, para que se distinga de la simple. En los modelos lineales y la SVM, las variables numéricas pasan por Yeo-Johnson antes de imputar, para que los modelos lineales del imputador trabajen sobre escalas menos asimétricas, y se estandarizan después. Los códigos binarios y ordinales imputados se redondean y se recortan al rango observado en el entrenamiento.

En los modelos lineales y la SVM, las ordinales solo se estandarizan, las binarias quedan en 0 y 1 y la temporada de inscripción va con codificación one-hot. Todo se ajusta dentro del `Pipeline`, con los datos de entrenamiento.

**Ponderación de los niveles.** Cada ajuste pondera las observaciones con
$$w_k = \left(\frac{n}{K\, n_k}\right)^{\alpha},$$
donde $n_k$ es la frecuencia del nivel $k$ en los datos de ese ajuste y $K$ el número de niveles presentes. Con $\alpha = 1$ se obtienen los pesos inversamente proporcionales a la frecuencia de la formulación; con $\alpha = 0{,}5$ la ponderación se atenúa. El exponente se elige en la validación interna porque las dos configuraciones de los experimentos iniciales trataron mal el nivel severo con la ponderación completa. En Frank y Hall cada binario pondera sus dos clases, y en la regresión cada observación lleva el peso de su nivel.

**Hiperparámetros y límites fijos.**
- *Modelo lineal:* `l1_ratio` en {0; 0,5; 1} elige el tipo de regularización (ridge, elastic net o lasso) y `C` o `alpha` su intensidad.
- *Logística:* `saga`, con hasta 10.000 iteraciones.
- *Elastic net:* hasta 10.000 iteraciones.
- *Bosque aleatorio:* 500 árboles.
- *Gradient boosting:* parada temprana con hasta 1.000 iteraciones, 20 iteraciones sin mejora y el 15 % de los datos de entrenamiento como validación.
- *Semillas:* todos los componentes con azar llevan semilla fija, incluidos el bosque, la parada temprana, el imputador, los pliegues de la calibración y los de los umbrales.

**Validación.** La validación externa es estratificada, con cinco particiones repetidas cinco veces, y la interna tiene tres particiones estratificadas dentro de cada partición de entrenamiento. El QWK de la validación interna elige el candidato de cada búsqueda. Las particiones externas se ajustan en paralelo y cada búsqueda interna es secuencial.

**Presupuesto de cómputo y recortes.** La búsqueda completa tiene un presupuesto de 10 horas de reloj. Antes de ejecutarla, la sección 3 mide los tiempos de la primera repetición. Con esos tiempos, la regla elige el nivel de recorte más bajo cuya proyección cabe en el presupuesto. Los niveles son acumulativos:
1. $\alpha$ solo en 1 para la SVM;
2. grilla de la SVM reducida a 4 × 4;
3. bosques de 300 árboles;
4. gradient boosting sin `l2_regularization`;
5. gradient boosting con búsqueda aleatoria de 40 candidatos, como último recurso.

La regla usa solo los tiempos medidos y nunca el QWK. Como los tiempos dependen del equipo en el que se ejecuta el notebook, el nivel elegido se reporta junto con los resultados y forma parte de la firma del caché.

**Procesos y memoria.** Los recursos se calculan a partir del equipo en el que se ejecuta el notebook:
- *Procesos por algoritmo:* cada partición externa se ajusta en un proceso. El modelo lineal, la SVM y gradient boosting usan todos los núcleos menos dos. El bosque aleatorio, que guarda cientos de árboles en memoria, usa un proceso por cada tres de esos núcleos.
- *Regla de memoria:* el presupuesto es el 75 % de la memoria física del equipo. Si la memoria máxima medida por proceso en la sección 3, multiplicada por un factor de seguridad de 1,25, no cabe en ese presupuesto, el número de procesos de ese algoritmo se reduce hasta que quepa.
- *Independencia de los resultados:* el número de procesos cambia el tiempo de ejecución, no los resultados, porque cada ajuste tiene semillas fijas.

**Convergencia.** Las advertencias de convergencia se cuentan en cada ajuste y se separan entre las del modelo y las del imputador. Si en la primera repetición más del 5 % de los ajustes de una configuración no converge, sus límites de iteraciones se multiplican por 5 antes de la búsqueda completa.

**Regla de selección.**
1. Se identifica la configuración con mayor QWK medio.
2. Entre las configuraciones cuya diferencia frente a ella no es significativa según la prueba t corregida de Nadeau y Bengio (unilateral, $\alpha = 0{,}05$), se selecciona la de menor tasa de subestimación. Dos tasas que difieren en menos de 0,01 se consideran empatadas.
3. Si persiste el empate, se elige la de menor complejidad.

El orden de complejidad se refiere a la capacidad del modelo final, no a su interpretabilidad ni al costo de ajustarlo:
- *Algoritmo* (lineal < SVM < bosque aleatorio < gradient boosting): el modelo lineal es una función lineal de los predictores; la SVM es una sola frontera suave controlada por dos hiperparámetros, aunque sea la menos interpretable; el bosque promedia cientos de árboles independientes; gradient boosting encadena árboles que corrigen los errores de los anteriores.
- *Formulación* (regresión con umbrales < multiclase < Frank y Hall): la regresión ajusta una sola función y tres escalares, pues su validación interna es parte del ajuste y no del modelo; la multiclase ajusta una función por nivel; Frank y Hall ajusta tres modelos independientes.
- *Estrategia de faltantes* (nativa = simple < iterativa): la iterativa agrega un modelo de imputación por variable. Como la estrategia se elige en cada partición, se usa la más frecuente de la configuración.

Como la selección usa las mismas particiones con las que se reporta el desempeño, el QWK de la configuración seleccionada tiene un sesgo optimista, que se declara al reportarlo.

**Revisión de los bordes de la grilla.** Si un hiperparámetro elige un extremo de su grilla en al menos 5 de las 25 particiones, la grilla se amplía un paso en esa dirección y esa configuración se repite y se guarda aparte, como máximo una vez. La regla usa solo los hiperparámetros elegidos en la validación interna.

**Diferencia mínima relevante y familias de pruebas.** Para afirmar que un conjunto de variables no aporta de forma relevante, la diferencia de QWK se contrasta contra $\Delta = 0{,}03$: el aporte es despreciable si el límite superior unilateral del 95 %, con el error estándar corregido, queda por debajo de $\Delta$. El valor se fijó como una fracción pequeña del QWK que alcanzan los modelos de este problema, por debajo de la cual una ganancia no cambiaría la utilidad de los indicadores físicos para un tamizaje. Los valores p se ajustan con el método de Holm dentro de cuatro familias:
- superioridad de H2 y H3;
- aporte despreciable de H2 y H3;
- comparaciones en la subpoblación con actigrafía;
- comparaciones pareadas del nivel severo.

La regla de selección no se ajusta por multiplicidad, como establece la formulación.

**Cachés.** Cada partición externa de cada evaluación se guarda en `data/interim/03_busqueda_modelos/cache/`, con una firma que incluye la configuración, la grilla completa, las columnas y los valores de los datos, la partición, las semillas y las versiones de scikit-learn y del caché. Al volver a ejecutar el notebook, solo se ajustan las particiones sin un archivo válido; `RECALCULAR = True` fuerza el recálculo. Aparte, `Pipeline(memory=...)` guarda el preprocesamiento ajustado para no repetirlo en cada candidato de la grilla, en `pipeline_memory/`.

In [7]:
# Candidatos de cada configuración en cada nivel de recorte, y árboles del bosque.
indice_configuraciones = pd.MultiIndex.from_tuples(
    [(NOMBRES_ALGORITMOS[a], NOMBRES_FORMULACIONES[f]) for a, f in CONFIGURACIONES],
    names=["algoritmo", "formulación"],
)
candidatos_por_nivel = pd.DataFrame(
    {
        f"nivel {nivel}": [n_candidates(a, f, nivel) for a, f in CONFIGURACIONES]
        for nivel in range(MAX_CUT_LEVEL + 1)
    },
    index=indice_configuraciones,
)
candidatos_por_nivel.loc[("bosque aleatorio", "árboles por bosque"), :] = [
    n_forest_trees(nivel) for nivel in range(MAX_CUT_LEVEL + 1)
]
display(candidatos_por_nivel.astype(int))

# Recursos de este equipo y procesos iniciales por algoritmo, antes de la regla de
# memoria, que solo puede reducirlos.
print(
    f"Equipo: {N_NUCLEOS} núcleos y {MEMORIA_TOTAL_GB:.1f} GB de memoria; "
    f"presupuesto de memoria para los procesos: "
    f"{memory_budget_gb(MEMORIA_TOTAL_GB):.1f} GB"
)
procesos_iniciales = pd.Series(
    {NOMBRES_ALGORITMOS[a]: available_workers(a, N_NUCLEOS) for a in ALGORITHMS},
    name="procesos en paralelo",
).rename_axis("algoritmo")
display(procesos_iniciales.to_frame())

# Decisiones fijadas en el paquete antes de la búsqueda. El tipo object conserva
# los enteros como enteros en la tabla.
decisiones = pd.Series(
    {
        "presupuesto de la búsqueda completa (horas)": SEARCH_BUDGET_HOURS,
        "núcleos reservados": RESERVED_CORES,
        "núcleos por proceso en el bosque aleatorio": FOREST_CORES_PER_WORKER,
        "fracción de la memoria para los procesos": MEMORY_BUDGET_FRACTION,
        "factor de seguridad de memoria": MEMORY_SAFETY_FACTOR,
        "máximo de ajustes sin convergencia": CONVERGENCE_WARNING_MAX_SHARE,
        "multiplicador de iteraciones": MAX_ITER_INCREASE,
        "iteraciones de los modelos lineales": LINEAR_MAX_ITER,
        "árboles del bosque (sin recorte)": FOREST_N_ESTIMATORS,
        "árboles del bosque (nivel 3)": CUT_FOREST_N_ESTIMATORS,
        "candidatos aleatorios de gradient boosting (nivel 5)": (
            CUT_BOOSTING_RANDOM_CANDIDATES
        ),
        "iteraciones máximas de gradient boosting": BOOSTING_MAX_ITER,
        "iteraciones sin mejora de gradient boosting": BOOSTING_N_ITER_NO_CHANGE,
        "fracción de validación de gradient boosting": BOOSTING_VALIDATION_FRACTION,
        "nivel de significancia de la regla de selección": SELECTION_ALPHA,
        "tolerancia de empate en la tasa de subestimación": (
            UNDERESTIMATION_TIE_TOLERANCE
        ),
        "particiones mínimas para ampliar un borde": EDGE_RULE_MIN_FOLDS,
        "diferencia mínima relevante de QWK (Δ)": MINIMUM_RELEVANT_QWK_DIFFERENCE,
        "semilla": RANDOM_SEED,
        "versión del caché": CACHE_VERSION,
    },
    name="valor",
    dtype=object,
).rename_axis("decisión")
display(decisiones.to_frame())

# Familias de pruebas con ajuste de Holm, con los nombres del paquete traducidos.
NOMBRES_FAMILIAS = {
    "superiority": "superioridad",
    "negligibility": "aporte despreciable",
    "actigraphy_subset": "subpoblación con actigrafía",
    "severe_paired": "comparaciones pareadas del nivel severo",
}
NOMBRES_PRUEBAS = {
    "H2": "H2: principal frente a referencia",
    "H3": "H3: principal frente a principal sin actigrafía",
    "reference_with_actigraphy_vs_reference": (
        "referencia con actigrafía frente a referencia"
    ),
    "main_vs_main_without_actigraphy": "principal frente a principal sin actigrafía",
    "selected_vs_best_of_each_other_algorithm": (
        "seleccionada frente a la mejor de cada otro algoritmo"
    ),
}
pd.Series(
    {
        NOMBRES_FAMILIAS[familia]: "; ".join(NOMBRES_PRUEBAS[p] for p in pruebas)
        for familia, pruebas in HOLM_FAMILIES.items()
    },
    name="pruebas",
).rename_axis("familia").to_frame()

nivel 0  nivel 1  nivel 2  \
algoritmo             formulación                                         
lineal regularizado   regresión con umbrales       72       72       72   
                      multiclase                   84       84       84   
                      Frank y Hall                 84       84       84   
SVM con kernel radial regresión con umbrales      240      120       64   
                      multiclase                  120       60       32   
                      Frank y Hall                120       60       32   
bosque aleatorio      regresión con umbrales       64       64       64   
                      multiclase                   64       64       64   
                      Frank y Hall                 64       64       64   
gradient boosting     regresión con umbrales      324      324      324   
                      multiclase                  324      324      324   
                      Frank y Hall                324      324      324   
bosque aleatorio      árboles por bosque          500      500      500   

                                              nivel 3  nivel 4  nivel 5  
algoritmo             formulación                                        
lineal regularizado   regresión con umbrales       72       72       72  
                      multiclase                   84       84       84  
                      Frank y Hall                 84       84       84  
SVM con kernel radial regresión con umbrales       64       64       64  
                      multiclase                   32       32       32  
                      Frank y Hall                 32       32       32  
bosque aleatorio      regresión con umbrales       64       64       64  
                      multiclase                   64       64       64  
                      Frank y Hall                 64       64       64  
gradient boosting     regresión con umbrales      324      108       40  
                      multiclase                  324      108       40  
                      Frank y Hall                324      108       40  
bosque aleatorio      árboles por bosque          300      300      300

Equipo: 15 núcleos y 24.0 GB de memoria; presupuesto de memoria para los procesos: 18.0 GB


,procesos en paralelo
algoritmo,
lineal regularizado,13
SVM con kernel radial,13
bosque aleatorio,4
gradient boosting,13


,valor
decisión,
presupuesto de la búsqueda completa (horas),10.0
núcleos reservados,2
núcleos por proceso en el bosque aleatorio,3
fracción de la memoria para los procesos,0.75
factor de seguridad de memoria,1.25
máximo de ajustes sin convergencia,0.05
multiplicador de iteraciones,5
iteraciones de los modelos lineales,10000
árboles del bosque (sin recorte),500


,pruebas
familia,
superioridad,H2: principal frente a referencia; H3: principal frente a principal sin actigrafía
aporte despreciable,H2: principal frente a referencia; H3: principal frente a principal sin actigrafía
subpoblación con actigrafía,referencia con actigrafía frente a referencia; principal frente a principal sin actigrafía
comparaciones pareadas del nivel severo,seleccionada frente a la mejor de cada otro algoritmo


### Conclusión

El diseño queda fijado antes de ejecutar la búsqueda.

- **Configuraciones.** Son doce, y cada una tiene como hiperparámetros la estrategia de faltantes y la intensidad de la ponderación, además de los del modelo.
- **Tamaño de las grillas.** Sin recortes, cada búsqueda interna evalúa entre 64 candidatos (bosque aleatorio) y 324 (gradient boosting). Los niveles de recorte reducen solo la SVM, el número de árboles del bosque y gradient boosting; el modelo lineal conserva su grilla completa en todos los niveles.
- **Procesos.** En este equipo, de 15 núcleos, se usan 12 procesos en paralelo para el modelo lineal, la SVM y gradient boosting, y 4 para el bosque aleatorio.
- **Reglas mecánicas.** La regla de memoria solo puede reducir estos números. Las reglas de recorte, convergencia y bordes se aplican según los tiempos, las advertencias y los hiperparámetros elegidos, nunca según el QWK.
- **Selección y pruebas.** La regla de selección, la diferencia mínima relevante y las familias de pruebas también están definidas de antemano.

La siguiente sección mide el costo de una repetición para decidir, con la regla de recorte, cómo se ejecuta la búsqueda completa dentro del presupuesto.